# Kapitel 19.7 - Fortgeschrittene Uebungen zu CGI und WSGI

Dieses Notebook ist als intensives Training gedacht.
Jede Aufgabe trainiert ein anderes Teilproblem: Routing, Validierung, Sicherheit, Fehlercodes, Antwortdesign.

# Lernziele

- komplexere Webaufgaben strukturiert loesen
- saubere Fehlerobjekte entwerfen
- WSGI-Routing und Query-Handling sicher anwenden
- eigene Entscheidungen fachlich begruenden

# Voraussetzungen

- Kapitel 19.1 bis 19.6
- gute Python-Grundlagen (Funktionen, Listen, Dictionaries, Fehlerbehandlung)

# Theorie

Fortgeschrittene Uebungen sind keine reinen Coding-Aufgaben.
Sie pruefen vor allem die Qualitaet deiner Entscheidungen:

- Wo validierst du?
- Welcher Statuscode ist korrekt?
- Wie sieht ein gutes Fehlerobjekt aus?

# Erklaerung

In diesem Notebook arbeiten wir im Muster:

Aufgabe -> Denkhinweise -> Referenzloesung -> Reflexion

So lernst du nicht nur das Ergebnis, sondern den Weg dorthin.

# Syntax

```python
def endpoint(environ, start_response):
    ...
```

# Merke

- Ein endpoint ist erst dann gut, wenn auch die Fehlerfaelle sauber sind.
- Konsistenz ist wichtiger als kreative Einzelloesungen.

# Parameter

- `PATH_INFO`
- `QUERY_STRING`
- optionale Felder wie `limit`, `offset`, `sort`

# Rueckgabewert

Alle Aufgaben arbeiten mit WSGI-konformen Antworten (bytes-Iterable).
Intern werden zur Vereinfachung auch Python-Dictionaries genutzt.

# Beispiel 1 - Aufgabe

Baue einen Endpoint `/api/gesundheit`, der JSON mit Feldern `status`, `zeitpunkt`, `dienst` liefert.

Anforderungen:
- Statuscode 200
- sauberes JSON
- UTF-8

In [ ]:
# Beispiel 1 - Referenzloesung
import json
from datetime import datetime

def json_response(start_response, data, status='200 OK'):
    body = json.dumps(data, ensure_ascii=False).encode('utf-8')
    start_response(status, [('Content-Type', 'application/json; charset=utf-8')])
    return [body]

def gesundheit_endpoint(environ, start_response):
    daten = {
        'status': 'ok',
        'zeitpunkt': datetime.now().isoformat(timespec='seconds'),
        'dienst': 'kursportal'
    }
    return json_response(start_response, daten)

# Beispiel 2 - Aufgabe

Baue einen Endpoint `/api/kurse?limit=...`:

- `limit` optional
- wenn gesetzt: numerisch und > 0
- bei Fehler: 400
- bei Erfolg: nur die ersten `limit` Kurse

In [ ]:
# Beispiel 2 - Referenzloesung
from urllib.parse import parse_qs

KURSE = [
    {'id': 1, 'titel': 'Python Grundlagen'},
    {'id': 2, 'titel': 'WSGI Praxis'},
    {'id': 3, 'titel': 'Fehlerbehandlung'},
    {'id': 4, 'titel': 'Testing Basics'}
]

def kurse_mit_limit(environ, start_response):
    query = parse_qs(environ.get('QUERY_STRING', ''))
    limit_text = query.get('limit', [''])[0].strip()

    if not limit_text:
        return json_response(start_response, {'kurse': KURSE})

    if not limit_text.isdigit() or int(limit_text) <= 0:
        return json_response(start_response, {'fehler': 'limit muss > 0 sein'}, '400 Bad Request')

    limit = int(limit_text)
    return json_response(start_response, {'kurse': KURSE[:limit]})

# Beispiel 3 - Aufgabe

Implementiere ein einheitliches Fehlerformat:

```json
{"ok": false, "code": "INVALID_INPUT", "nachricht": "..."}
```

Nutze es fuer mehrere Fehlerfaelle.

In [ ]:
# Beispiel 3 - Referenzloesung
def fehler_response(start_response, code, nachricht, status='400 Bad Request'):
    return json_response(
        start_response,
        {'ok': False, 'code': code, 'nachricht': nachricht},
        status
    )

def beispiel_endpoint(environ, start_response):
    query = parse_qs(environ.get('QUERY_STRING', ''))
    id_text = query.get('id', [''])[0]

    if not id_text:
        return fehler_response(start_response, 'MISSING_ID', 'id fehlt')

    if not id_text.isdigit():
        return fehler_response(start_response, 'INVALID_INPUT', 'id muss numerisch sein')

    return json_response(start_response, {'ok': True, 'id': int(id_text)})

# Praxisbeispiel

Kombiniere die drei Muster zu einer kompakten API mit den Routen:
- `/api/gesundheit`
- `/api/kurse`
- `/api/kurs?id=...`

Achte auf konsistente Fehlerobjekte und Statuscodes.

# Haeufige Fehler

1. Fehlerobjekte je Endpoint unterschiedlich strukturieren.
2. 200 senden, obwohl ein Fehler vorliegt.
3. Numerische Felder nicht als Integer behandeln.
4. Query-Parameter ohne Defaults lesen.

# Best Practice

- Definiere ein kleines API-Styleguide fuer deine Endpunkte.
- Halte Utility-Funktionen zentral (`json_response`, `fehler_response`).
- Fuehre frueh einfache automatisierte Tests ein.

# Tipp

Fuer schnelle Selbstkontrolle kannst du pro Endpoint drei Testfaelle planen:
- Happy Path
- Validierungsfehler
- Ressourcenfehler (nicht gefunden)

# Uebung

Implementiere eine Route `/api/suche?text=...&limit=...` mit folgenden Regeln:
- `text` Pflichtfeld
- `limit` optional, default 10
- Fehlerobjekte wie oben
- Ergebnisfeld `treffer` als Liste

In [ ]:
# Loesung
def suche_endpoint(environ, start_response):
    query = parse_qs(environ.get('QUERY_STRING', ''))
    text = query.get('text', [''])[0].strip().lower()
    limit_text = query.get('limit', ['10'])[0].strip()

    if not text:
        return fehler_response(start_response, 'MISSING_TEXT', 'text ist erforderlich')

    if not limit_text.isdigit() or int(limit_text) <= 0:
        return fehler_response(start_response, 'INVALID_LIMIT', 'limit muss > 0 sein')

    limit = int(limit_text)
    treffer = [kurs for kurs in KURSE if text in kurs['titel'].lower()][:limit]
    return json_response(start_response, {'ok': True, 'treffer': treffer})

# Zusammenfassung

Dieses Training hat den Fokus auf fortgeschrittene Qualitaetskriterien gelegt:
- konsistente Fehlerbehandlung
- klare Statuscodes
- robuste Query-Verarbeitung

Damit bist du gut auf echte API-Projekte vorbereitet.

# Weiterfuehrende Links

- RFC 9110 (HTTP Semantics)
- Python `wsgiref`
- API Design Best Practices

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

HTTP Semantics
Status Code Family
WSGI Callable
Request Lifecycle
Input Sanitization
Header Validation

In [ ]:
# WSGI-Minibeispiel mit Statuscode
def app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    if path == "/health":
        start_response("200 OK", [("Content-Type", "text/plain")])
        return [b"ok"]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.